# 03 — Modélisation, GridSearchCV & tracking MLflow

Ce notebook couvre **Milestone 1 & 4**.

- Setup MLflow
- Pipelines anti data leakage (preprocessing + SMOTE + modèle)
- GridSearchCV sur **score métier**
- Logging des résultats et artefacts (ROC, confusion matrix, modèle)

In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

import sys, os
sys.path.insert(0, os.path.abspath(".."))  # monte d'un niveau (../)
from src.config import PATHS
from src.data import load_parquet
from src.metrics import optimal_threshold_cost, get_proba
from src.models import make_cv, gridsearch_dummy, gridsearch_lgbm_smote, gridsearch_xgb_smote, gridsearch_logreg_smote, gridsearch_rf_smote, run_gridsearch

from src.mlflow_utils import init_mlflow, track_run
import mlflow

## 1) Charger dataset préparé

In [2]:
df = load_parquet(PATHS.data_processed / "train_fe.parquet")

assert "TARGET" in df.columns, "Le dataset doit contenir TARGET (uniquement train)."
X = df.drop(columns=["TARGET"])
y = df["TARGET"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print(set(X.dtypes))
print(X.select_dtypes(include='object').columns.tolist())



(79999, 125) (20000, 125)
{dtype('float64'), dtype('bool'), dtype('O'), dtype('int64')}
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


## 2) Init MLflow

In [11]:
from datetime import datetime

experiment_name = "credit_scoring"
init_mlflow()

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment(experiment_name)
# mlflow.enable_system_metrics_logging()
# mlflow.config.set_system_metrics_sampling_interval(1)

notebook_run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print("notebook_run_id: ", notebook_run_id)


notebook_run_id:  20260514_115127


In [4]:
cv = make_cv(2)

In [5]:
from src.models import build_preprocessor

print(X.shape, set(X.dtypes))

pre = build_preprocessor(X)
Xt = pre.fit_transform(X)
print(Xt.shape, Xt.dtype)
print(type(Xt))

(99999, 125) {dtype('float64'), dtype('bool'), dtype('O'), dtype('int64')}
(99999, 256) float64
<class 'numpy.ndarray'>


In [6]:
from sklearn.metrics import roc_auc_score
def get_auc_metric(gs, best_model, X_train, y_train, X_test, y_test):
    # 1. AUC TRAIN
    y_train_proba = get_proba(best_model, X_train)
    auc_train = roc_auc_score(y_train, y_train_proba)
    
    # 2. AUC CROSS-VALIDATION
    auc_cv = gs.cv_results_["mean_test_AUC"][gs.best_index_]
    
    # optionnel : écart-type CV
    auc_cv_std = gs.cv_results_["std_test_AUC"][gs.best_index_]
    
    # 3. AUC VALIDATION / TEST
    y_test_proba = get_proba(best_model, X_test)
    auc_test = roc_auc_score(y_test, y_test_proba)

    return {
        "AUC_train": auc_train,
        "AUC_cv": auc_cv,
        "AUC_cv_std": auc_cv_std,
        "AUC_test": auc_test
    }

## 3) LightGBM

In [7]:
pipe, param_grid = gridsearch_lgbm_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_lightGBM = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_lightGBM, X_test)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_test.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

auc_all = get_auc_metric(gs, best_model_lightGBM, X_train, y_train, X_test, y_test)

print(gs.cv_results_.keys())

extra_metrics_lightGBM = {
    # "AUC_test": roc_auc_score(y_test, y_proba),
    "AUC_train": auc_all["AUC_train"],
    "AUC_cv": auc_all["AUC_cv"],
    "AUC_cv_std": auc_all["AUC_cv_std"],
    "AUC_test": auc_all["AUC_test"],
    
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
    
    "fit_time": fit_time,
    "predict_time": predict_time
}


Fitting 2 folds for each of 8 candidates, totalling 16 fits
[LightGBM] [Info] Number of positive: 73525, number of negative: 73525
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.920723
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.659486
[LightGBM] [Debug] init for col-wise cost 0.209366 seconds, init for row-wise cost 0.506493 seconds
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.256718 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Debug] Using Sparse Multi-Val Bin
[LightGBM] [Info] Total Bins 54600
[LightGBM] [Info] Number of data points in the train set: 147050, number of used features: 238
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 10
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 10

C:\apps\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\apps\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\apps\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


dict_keys(['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time', 'param_model__colsample_bytree', 'param_model__learning_rate', 'param_model__max_depth', 'param_model__min_child_samples', 'param_model__n_estimators', 'param_model__num_leaves', 'param_model__subsample', 'params', 'split0_test_business', 'split1_test_business', 'mean_test_business', 'std_test_business', 'rank_test_business', 'split0_test_threshold', 'split1_test_threshold', 'mean_test_threshold', 'std_test_threshold', 'rank_test_threshold', 'split0_test_AUC', 'split1_test_AUC', 'mean_test_AUC', 'std_test_AUC', 'rank_test_AUC', 'split0_test_f1', 'split1_test_f1', 'mean_test_f1', 'std_test_f1', 'rank_test_f1'])


### Résultat

In [8]:

gs.best_params_, gs.best_score_, extra_metrics_lightGBM

({'model__colsample_bytree': 0.8,
  'model__learning_rate': 0.05,
  'model__max_depth': 10,
  'model__min_child_samples': 60,
  'model__n_estimators': 400,
  'model__num_leaves': 31,
  'model__subsample': 0.8},
 np.float64(-21595.0),
 {'AUC_train': 0.8901279651076253,
  'AUC_cv': np.float64(0.7480473401459309),
  'AUC_cv_std': np.float64(0.002510086151148505),
  'AUC_test': 0.7570200907367388,
  'business_cost_test': 10519.0,
  'business_threshold_test': 0.10999999999999999,
  'business_score_cv': np.float64(-21595.0),
  'mean_cv_threshold': np.float64(0.0875),
  'fit_time': 247.79630160331726,
  'predict_time': 0.9759376049041748})

### MLFlow tracking

In [9]:
run_name = "lightgbm"
model_type = "boosting"

track_run(
    run_name= run_name,
    model=best_model_lightGBM,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    params=gs.best_params_,
    extra_metrics=extra_metrics_lightGBM,
    y_test_proba=y_proba,
    y_test_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
    model_type=model_type,
)

C:\apps\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/13 17:44:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


C:\apps\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\apps\anaconda3\Lib\site-packages\mlflow\tracking\_model_registry\utils.py:215: FutureWarning: The filesystem model registry backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance.
  return FileStore(store_uri)
Registered model 'lightgbm_LGBMClassifier' already exists. Creating a new version of this model...
Created version '9' of model 'lightgbm_LGBMClassifier'.


### SAVE model to .joblib

#### MLFlow Model Repository: récupération du de la dernière version du modèle

In [18]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

def get_model_from_mlflow_model_registry(experiment_name, run_name):
    mlflow.set_tracking_uri("file:./mlruns")
    
    # experiment_name = "credit_scoring"
    wanted_run_name = run_name
    
    client = MlflowClient()
    experiment = client.get_experiment_by_name(experiment_name)
    
    if experiment is None:
        raise ValueError(f"Experiment introuvable : {experiment_name}")
    
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.mlflow.runName = '{wanted_run_name}'",
        order_by=["start_time DESC"],
        max_results=1
    )
    
    if not runs:
        raise ValueError(f"Aucun run trouvé avec le nom : {wanted_run_name}")
    
    run = runs[0]
    run_id = run.info.run_id
    
    print("Run récupéré :", run_id)
    print("Nom du run :", run.data.tags.get("mlflow.runName"))
    print("Date début :", run.info.start_time)
    
    best_model = mlflow.sklearn.load_model(
        f"runs:/{run_id}/{wanted_run_name}_model"
    )
    
    print("Data récupérée dans MLFlow")
    
    threshold = float(run.data.params.get("threshold", 0.5))
    
    threshold_info = {
        "threshold": threshold,
        "run_id": run_id,
        "run_name": run.data.tags.get("mlflow.runName"),
        "model_type": run.data.tags.get("model_type"),
        "notebook_run_id": run.data.tags.get("notebook_run_id"),
    }
    
    artifact = {
        "model": best_model,
        "threshold": threshold,
        "threshold_info": threshold_info,
        "score_name": "business_cost_opt",
    }


In [19]:
import joblib

# best_model = best_model_lightGBM #gs.best_estimator_

# artifact = {
#     "model": best_model,
#     "threshold": threshold_info["threshold"],
#     "threshold_info": threshold_info,
#     "score_name": "business_cost_opt",
# }

artifact = get_model_from_mlflow_model_registry(experiment_name, run_name)

# joblib.dump(best_model, "../model/model.joblib")
joblib.dump(artifact, "../model/best_model_lightGBM.joblib")

print("✅ Model saved to model.joblib")
print("Best params:", gs.best_params_)
print("Best score:", gs.best_score_)

Run récupéré : 22f57a0ec2e74e808fe239bfc8f24302
Nom du run : lightgbm
Date début : 1778687022603


Data récupérée dans MLFlow
✅ Model saved to model.joblib
Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 10, 'model__min_child_samples': 60, 'model__n_estimators': 400, 'model__num_leaves': 31, 'model__subsample': 0.8}
Best score: -21595.0


## 4) Baseline DummyClassifier (référence)

In [11]:
pipe, param_grid = gridsearch_dummy(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_dummyClassif = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_dummyClassif, X_test)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_test.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

# CV results artifact (utile pour stabilité & soutenance)
cv_results_df = pd.DataFrame(gs.cv_results_)

auc_all = get_auc_metric(best_model_dummyClassif, X_train, y_train, X_test, y_test)

extra_metrics_dummyClassif = {
    # "AUC_test": roc_auc_score(y_test, y_proba),
    "AUC_train": auc_all["AUC_train"],
    "AUC_cv": auc_all["AUC_cv"],
    "AUC_cv_std": auc_all["AUC_cv_std"],
    "AUC_test": auc_all["AUC_test"],
    
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
    
    "fit_time": fit_time,
    "predict_time": predict_time
}


Fitting 2 folds for each of 1 candidates, totalling 2 fits


### Résultat

In [12]:
gs.best_params_, gs.best_score_, extra_metrics_dummyClassif

({},
 np.float64(-32370.0),
 {'AUC_train': 0.5,
  'AUC_cv': np.float64(0.5),
  'AUC_cv_std': np.float64(0.0),
  'AUC_test': 0.5,
  'business_cost_test': 16190.0,
  'business_threshold_test': 0.05,
  'business_score_cv': np.float64(-32370.0),
  'mean_cv_threshold': np.float64(0.05),
  'fit_time': 5.85222601890564,
  'predict_time': 0.15426015853881836})

### MLFlow tracking

In [13]:
run_name = "dummy_baseline"
model_type = "dummy"

track_run(
    run_name=run_name,
    model=best_model_dummyClassif,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    params=gs.best_params_,
    extra_metrics=extra_metrics_dummyClassif,
    y_test_proba=y_proba,
    y_test_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
    model_type=model_type,
)

C:\apps\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/13 17:21:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Registered model 'dummy_baseline_DummyClassifier' already exists. Creating a new version of this model...
Created version '5' of model 'dummy_baseline_DummyClassifier'.


## 5) XGBoost 

In [14]:
pipe, param_grid = gridsearch_xgb_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_xgBoost = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_xgBoost, X_test)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_test.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

auc_all = get_auc_metric(best_model_xgBoost, X_train, y_train, X_test, y_test)

extra_metrics_xgBoost = {
    # "AUC_test": roc_auc_score(y_test, y_proba),
    "AUC_train": auc_all["AUC_train"],
    "AUC_cv": auc_all["AUC_cv"],
    "AUC_cv_std": auc_all["AUC_cv_std"],
    "AUC_test": auc_all["AUC_test"],
    
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
    
    "fit_time": fit_time,
    "predict_time": predict_time
}


Fitting 2 folds for each of 2 candidates, totalling 4 fits


### Résultat

In [15]:
gs.best_params_, gs.best_score_, extra_metrics_xgBoost

({'model__colsample_bytree': 0.8,
  'model__learning_rate': 0.05,
  'model__max_depth': 4,
  'model__min_child_weight': 5,
  'model__n_estimators': 500,
  'model__subsample': 0.8},
 np.float64(-21790.5),
 {'AUC_train': 0.8125713998199794,
  'AUC_cv': np.float64(0.7490062466170664),
  'AUC_cv_std': np.float64(0.0020010761190015214),
  'AUC_test': 0.7583113037440741,
  'business_cost_test': 10349.0,
  'business_threshold_test': 0.105,
  'business_score_cv': np.float64(-21790.5),
  'mean_cv_threshold': np.float64(0.0925),
  'fit_time': 254.5007667541504,
  'predict_time': 0.19353175163269043})

### MLFlow tracking

In [16]:
run_name = "xgboost"
model_type = "boosting"

track_run(
    run_name=run_name,
    model=best_model_xgBoost,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    params=gs.best_params_,
    extra_metrics=extra_metrics_xgBoost,
    y_test_proba=y_proba,
    y_test_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
    model_type=model_type,
)


C:\apps\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/13 17:25:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Registered model 'xgboost_XGBClassifier' already exists. Creating a new version of this model...
Created version '5' of model 'xgboost_XGBClassifier'.


### SAVE model to .joblib

In [17]:
import joblib

# best_model = best_model_xgBoost #gs.best_estimator_

# artifact = {
#     "model": best_model,
#     "threshold": threshold_info["threshold"],
#     "threshold_info": threshold_info,
#     "score_name": "business_cost_opt",
# }

artifact = get_model_from_mlflow_model_registry(experiment_name, run_name)

# joblib.dump(best_model, "../model/model.joblib")
joblib.dump(artifact, "../model/best_model_xgBoost.joblib")

print("✅ Model saved to model.joblib")
print("Best params:", gs.best_params_)
print("Best score:", gs.best_score_)

✅ Model saved to model.joblib
Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 4, 'model__min_child_weight': 5, 'model__n_estimators': 500, 'model__subsample': 0.8}
Best score: -21790.5


In [18]:
# !pip freeze > requirements.txt

In [19]:
import sklearn
print(sklearn.__version__)

1.7.2


In [20]:
import imblearn
print(imblearn.__version__)

0.14.0


## 6) Logistic Regression + SMOTE

In [21]:
pipe, param_grid = gridsearch_logreg_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_logRegression = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_logRegression, X_test)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_test.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

auc_all = get_auc_metric(best_model_logRegression, X_train, y_train, X_test, y_test)

extra_metrics_logRegression = {
    # "AUC_test": roc_auc_score(y_test, y_proba),
    "AUC_train": auc_all["AUC_train"],
    "AUC_cv": auc_all["AUC_cv"],
    "AUC_cv_std": auc_all["AUC_cv_std"],
    "AUC_test": auc_all["AUC_test"],
    
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
    
    "fit_time": fit_time,
    "predict_time": predict_time
}


Fitting 2 folds for each of 2 candidates, totalling 4 fits


C:\apps\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Résultats

In [22]:
gs.best_params_, gs.best_score_, extra_metrics_logRegression

({'model__C': 1, 'model__solver': 'lbfgs'},
 np.float64(-27455.0),
 {'AUC_train': 0.6316057103679542,
  'AUC_cv': np.float64(0.6301731618636626),
  'AUC_cv_std': np.float64(0.003599893275295807),
  'AUC_test': 0.6348064855621551,
  'business_cost_test': 13714.0,
  'business_threshold_test': 0.5299999999999999,
  'business_score_cv': np.float64(-27455.0),
  'mean_cv_threshold': np.float64(0.5125),
  'fit_time': 302.0430557727814,
  'predict_time': 0.21309685707092285})

### MLFlow tracking

In [23]:
run_name = "logreg_smote"
model_type = "classic"

track_run(
    run_name=run_name,
    model=best_model_logRegression,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    params=gs.best_params_,
    extra_metrics=extra_metrics_logRegression,
    y_test_proba=y_proba,
    y_test_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
    model_type=model_type,
)

C:\apps\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/13 17:30:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Registered model 'logreg_smote_LogisticRegression' already exists. Creating a new version of this model...
Created version '5' of model 'logreg_smote_LogisticRegression'.


In [24]:
import mlflow

print("tracking uri =", mlflow.get_tracking_uri())

exp = mlflow.get_experiment_by_name("credit_scoring")
print("credit_scoring experiment =", exp)

runs = mlflow.search_runs()
print("Nb runs =", len(runs))
print(runs[["run_id", "experiment_id", "tags.mlflow.runName"]].tail(20))

tracking uri = file:./mlruns
credit_scoring experiment = <Experiment: artifact_location='file:C:/projects/openclassrooms_DS/P7/ds_project_7_credit_scoring/notebooks/mlruns/239362865412336589', creation_time=1778573919997, experiment_id='239362865412336589', last_update_time=1778573919997, lifecycle_stage='active', name='credit_scoring', tags={'mlflow.experimentKind': 'custom_model_development'}>
Nb runs = 27
                              run_id       experiment_id tags.mlflow.runName
7   45def029cf5947b3a0816cc9379e3d51  239362865412336589            rf_smote
8   322524fe57834bf1bb16cb01f3b0029d  239362865412336589        logreg_smote
9   dc82e884941f4928b613c4b582ba4102  239362865412336589             xgboost
10  ff796e9b3e24468293291cc3a031bad3  239362865412336589      dummy_baseline
11  c412d799241d40da8182a90a5fd90774  239362865412336589            lightgbm
12  360348bb2dfd4ce7ac520e18813f68db  239362865412336589            rf_smote
13  eaf474f4a1c14af6826d27e196160d49  23936286541

## 7) RandomForest + SMOTE

In [25]:
pipe, param_grid = gridsearch_rf_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_randomForest = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_randomForest, X_test)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_test.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

auc_all = get_auc_metric(best_model_randomForest, X_train, y_train, X_test, y_test)

extra_metrics_randomForest = {
    # "AUC_test": roc_auc_score(y_test, y_proba),
    "AUC_train": auc_all["AUC_train"],
    "AUC_cv": auc_all["AUC_cv"],
    "AUC_cv_std": auc_all["AUC_cv_std"],
    "AUC_test": auc_all["AUC_test"],
    
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
    
    "fit_time": fit_time,
    "predict_time": predict_time
}


Fitting 2 folds for each of 1 candidates, totalling 2 fits


### Résultats

In [26]:
gs.best_params_, gs.best_score_, extra_metrics_randomForest

({'model__max_depth': 8,
  'model__max_features': 'sqrt',
  'model__min_samples_leaf': 20,
  'model__n_estimators': 200},
 np.float64(-26205.0),
 {'AUC_train': 0.6899070747457698,
  'AUC_cv': np.float64(0.6662613659396845),
  'AUC_cv_std': np.float64(0.009498076786873144),
  'AUC_test': 0.6641772886368318,
  'business_cost_test': 12917.0,
  'business_threshold_test': 0.30499999999999994,
  'business_score_cv': np.float64(-26205.0),
  'mean_cv_threshold': np.float64(0.2825),
  'fit_time': 55.36084246635437,
  'predict_time': 0.31259703636169434})

### MLFlow tracking

In [27]:
run_name = "rf_smote"
model_type = "classic"

track_run(
    run_name=run_name,
    model=best_model_randomForest,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    params=gs.best_params_,
    extra_metrics=extra_metrics_randomForest,
    y_test_proba=y_proba,
    y_test_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
    model_type=model_type,
)

C:\apps\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/13 17:32:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Registered model 'rf_smote_RandomForestClassifier' already exists. Creating a new version of this model...
Created version '5' of model 'rf_smote_RandomForestClassifier'.


## Analyses des résultats
- Comparer via MLflow UI (runs)